# Destek 01 — Açılış sayıları (Görsel 1: Büyük yer değiştirme)

**Amaç:** Görsel 1'in iki grafiğini besleyen CSV'leri üretmek ve kontrol listesindeki #1, #2, #3 sayılarını hesaplamak.

**Girdi:** `data/clean/countries_clean.csv` (237 ülke, 2101 dışarıda; nüfus bin kişi → milyona çevrilir). Grafik B için `data/raw/` altında BM Medium dosyası (bölge satırları) varsa o kullanılır.

**Çıktı:** `data/out/g01_bolge_pay.csv`, `data/out/g01_medyan_yas_sira.csv` (ham dosya yoksa `_taslak` eki), bulgu cümleleri.

In [1]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
OUT = ROOT / "data/out"; OUT.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(ROOT / "data/clean/countries_clean.csv")
df = df[df.Time <= 2100].copy()
assert df.LocID.nunique() == 237 and df.Time.min() == 1950 and df.Time.max() == 2100
print("Ülke:", df.LocID.nunique(), "| yıl:", df.Time.min(), "-", df.Time.max(), "| satır:", len(df))

Ülke: 237 | yıl: 1950 - 2100 | satır: 35787


## Grafik A — Bölgelerin nüfus payı

In [2]:
reg = (df.groupby(["Region", "Time"], as_index=False).TPopulation1July.sum()
         .rename(columns={"Region": "region", "Time": "year"}))
reg["pop_millions"] = (reg.TPopulation1July / 1000).round(2)
reg["share_pct"] = (reg.TPopulation1July / reg.groupby("year").TPopulation1July.transform("sum") * 100).round(2)
reg["is_projection"] = reg.year >= 2024
g01a = reg[["region", "year", "pop_millions", "share_pct", "is_projection"]].sort_values(["region", "year"]).reset_index(drop=True)

tot = df.groupby("Time").TPopulation1July.sum()
assert abs(g01a.groupby("year").share_pct.sum() - 100).max() < 0.05, "Paylar 100 etmiyor"
assert g01a.region.nunique() == 6
g01a.to_csv(OUT / "g01_bolge_pay.csv", index=False)
print("Yazıldı:", OUT / "g01_bolge_pay.csv", g01a.shape)
sh = g01a.pivot(index="year", columns="region", values="share_pct")
display(sh.loc[[1950, 2024, 2100]].T)

Yazıldı: /home/claude/repo/data/out/g01_bolge_pay.csv (906, 5)


year,1950,2024,2100
region,,,
Africa,9.14,18.56,37.46
Asia,54.87,58.89,45.31
Europe,22.02,9.13,5.82
Latin America and the Caribbean,6.73,8.13,6.03
Northern America,6.74,4.72,4.67
Oceania,0.50,0.56,0.72


In [3]:
ctrl = pd.DataFrame([
 ("#1 Dünya nüfusu zirve yılı",   int(tot.idxmax()),                 2084),
 ("#1 Zirve nüfusu (milyar)",     round(tot.max()/1e6, 2),           10.29),
 ("#2 Afrika payı 1950 (%)",      round(sh.loc[1950, "Africa"], 1),  9.1),
 ("#2 Afrika payı 2024 (%)",      round(sh.loc[2024, "Africa"], 1),  18.6),
 ("#2 Afrika payı 2100 (%)",      round(sh.loc[2100, "Africa"], 1),  37.5),
 ("   Avrupa payı 1950 (%)",      round(sh.loc[1950, "Europe"], 1),  22.0),
 ("   Avrupa payı 2024 (%)",      round(sh.loc[2024, "Europe"], 1),  9.1),
 ("   Avrupa payı 2100 (%)",      round(sh.loc[2100, "Europe"], 1),  5.8),
], columns=["sayı", "hesaplanan", "yol_haritası_kontrol_değeri"])
ctrl["fark_%"] = ((ctrl.hesaplanan - ctrl.yol_haritası_kontrol_değeri) / ctrl.yol_haritası_kontrol_değeri * 100).round(2)
ctrl["uyum"] = ctrl["fark_%"].abs() <= 0.5
display(ctrl)

,sayı,hesaplanan,yol_haritası_kontrol_değeri,fark_%,uyum
0,#1 Dünya nüfusu zirve yılı,2084.00,2084.00,0.0,True
1,#1 Zirve nüfusu (milyar),10.29,10.29,0.0,True
2,#2 Afrika payı 1950 (%),9.10,9.10,0.0,True
3,#2 Afrika payı 2024 (%),18.60,18.60,0.0,True
4,#2 Afrika payı 2100 (%),37.50,37.50,0.0,True
5,Avrupa payı 1950 (%),22.00,22.00,0.0,True
6,Avrupa payı 2024 (%),9.10,9.10,0.0,True
7,Avrupa payı 2100 (%),5.80,5.80,0.0,True


**Çapraz kontrol (BM bölge satırları):** Ülke toplamlarından kurduğum bölge nüfusları, BM'nin kendi bölge satırlarıyla karşılaştırılır. Bu, `Region` atamasının doğru olduğunu da gösterir.

In [4]:
_f = next((p for p in (ROOT / "data/raw").glob("*") if p.is_file() and "medium" in p.name.lower()), None)
if _f is None:
    print("Ham Medium dosyası data/raw içinde yok; çapraz kontrol atlandı.")
else:
    _m = pd.read_csv(_f, usecols=["Location", "LocTypeName", "Time", "TPopulation1July"], encoding="utf-8-sig", low_memory=False)
    _m = _m[(_m.LocTypeName == "Geographic region") & (_m.Time <= 2100) & (_m.Location != "Americas")]
    un = _m.pivot(index="Time", columns="Location", values="TPopulation1July")
    mine = reg.pivot(index="year", columns="region", values="TPopulation1July")   # yuvarlanmamış, bin kişi
    fark_pop = (mine[un.columns] - un).abs().max()
    print("En büyük mutlak fark (bin kişi; yuvarlama düzeyinde olmalı):")
    display(fark_pop.round(3).to_frame("max_fark_bin_kisi"))
    assert (fark_pop < 1).all(), "Bölge toplamları BM bölge satırlarıyla tutmuyor"

En büyük mutlak fark (bin kişi; yuvarlama düzeyinde olmalı):


,max_fark_bin_kisi
region,
Africa,0.007
Asia,0.008
Europe,0.011
Latin America and the Caribbean,0.008
Northern America,0.002
Oceania,0.004


## Grafik B — Bölgelerin medyan yaş sıralaması (bump)

**Önemli:** Bölge medyan yaşı, ülke medyanlarının ortalaması değildir. Doğrusu BM Medium dosyasındaki bölge satırlarıdır (`LocTypeName = Geographic region`). Bu dosya repoda yok (`data/raw/`, herkes kendi bilgisayarında tutuyor). Aşağıdaki hücre ham dosyayı arar. Bulursa BM'nin kendi bölge satırlarıyla **final** dosyayı üretir. Bulamazsa **nüfus ağırlıklı ortalamayla TASLAK** üretir ve dosya adına `_taslak` ekler; taslak sunuma girmez. (BM tablosunda 7. bir "Americas" satırı da var; Latin Amerika + Kuzey Amerika toplamı olduğu için dışarıda bırakılır.)

In [5]:
REGIONS = ["Africa", "Asia", "Europe", "Latin America and the Caribbean", "Northern America", "Oceania"]
YEARS = list(range(1950, 2101, 10))

def find_raw(keyword):
    # BM dosyası uzantısız ya da .csv/.gz/.zip olabilir; adında anahtar kelime geçen ilk dosyayı al
    hits = sorted(p for p in (ROOT / "data/raw").glob("*") if p.is_file() and keyword in p.name.lower())
    return hits[0] if hits else None

def from_raw():
    f = find_raw("demographic_indicators_medium") or find_raw("medium")
    if f is None: return None
    use = ["Location", "LocTypeName", "Time", "MedianAgePop"]
    m = pd.read_csv(f, usecols=use, low_memory=False, encoding="utf-8-sig")    # yalnızca gereken sütunlar
    m = m[(m.LocTypeName == "Geographic region") & m.Location.isin(REGIONS) & (m.Time <= 2100)]
    assert m.Location.nunique() == 6, "6 bölge bulunamadı: sütun/etiket adlarını kontrol et"
    print("BM bölge satırları okundu:", f.name)
    return m.rename(columns={"Location": "region", "Time": "year", "MedianAgePop": "median_age"})[["region", "year", "median_age"]]

ma = from_raw()
kaynak = "bm_bolge_satiri"
if ma is None:
    kaynak = "taslak_nufus_agirlikli"
    ma = (df.groupby(["Region", "Time"]).apply(lambda g: np.average(g.MedianAgePop, weights=g.TPopulation1July))
            .rename("median_age").reset_index().rename(columns={"Region": "region", "Time": "year"}))
print("Medyan yaş kaynağı:", kaynak)

ma["median_age"] = ma.median_age.round(2)
yearly = ma.pivot(index="year", columns="region", values="median_age")
fark = yearly["Latin America and the Caribbean"] - yearly["Europe"]
gecis = int(fark[fark > 0].index.min()) if (fark > 0).any() else None
print("Latin Amerika > Avrupa ilk yıl:", gecis)
print("2100: Latin Amerika", yearly.loc[2100, "Latin America and the Caribbean"], "| Avrupa", yearly.loc[2100, "Europe"])
display(yearly.loc[[2080, 2084, 2085, 2090, 2095, 2100]].T)

g01b = ma[ma.year.isin(YEARS)].copy()
g01b["rank"] = g01b.groupby("year").median_age.rank(ascending=False, method="min").astype(int)   # 1 = en yaşlı
g01b["is_projection"] = g01b.year >= 2024
g01b = g01b[["region", "year", "median_age", "rank", "is_projection"]].sort_values(["year", "rank"]).reset_index(drop=True)
fn = "g01_medyan_yas_sira.csv" if kaynak == "bm_bolge_satiri" else "g01_medyan_yas_sira_taslak.csv"
g01b.to_csv(OUT / fn, index=False); print("Yazıldı:", OUT / fn, g01b.shape)
display(g01b.pivot(index="year", columns="region", values="rank"))

BM bölge satırları okundu: WPP2024_Demographic_Indicators_Medium_csv
Medyan yaş kaynağı: bm_bolge_satiri
Latin Amerika > Avrupa ilk yıl: 2084
2100: Latin Amerika 49.09 | Avrupa 48.62


year,2080,2084,2085,2090,2095,2100
region,,,,,,
Africa,31.11,31.95,32.16,33.17,34.14,35.06
Asia,45.00,45.35,45.45,45.93,46.41,46.77
Europe,47.58,47.48,47.49,47.75,48.22,48.62
Latin America and the Caribbean,47.06,47.56,47.68,48.23,48.70,49.09
Northern America,44.42,44.64,44.71,45.05,45.34,45.50
Oceania,40.49,40.84,40.94,41.41,41.92,42.43


Yazıldı: /home/claude/repo/data/out/g01_medyan_yas_sira.csv (96, 5)


region,Africa,Asia,Europe,Latin America and the Caribbean,Northern America,Oceania
year,,,,,,
1950,6,4,2,5,1,3
1960,5,4,1,6,2,3
1970,6,4,1,5,2,3
1980,6,4,1,5,2,3
1990,6,4,1,5,2,3
2000,6,4,1,5,2,3
2010,6,4,1,5,2,3
2020,6,4,1,5,2,3
2030,6,4,1,5,2,3


In [6]:
oldest_2100 = g01b[(g01b.year == 2100) & (g01b["rank"] == 1)].region.iloc[0]
print("2100'de en yaşlı bölge:", oldest_2100)
print()
print("BULGU (g01 A):  Afrika'nın dünya nüfusundaki payı %{:.1f} (1950) → %{:.1f} (2024) → %{:.1f} (2100).".format(sh.loc[1950,'Africa'], sh.loc[2024,'Africa'], sh.loc[2100,'Africa']))
print("BULGU (g01 B):  2100'de en yaşlı bölge {} (kaynak: {}).".format(oldest_2100, kaynak))

ok = "Hesaplandı, kontrol bekliyor"
kontrol_listesi = pd.DataFrame([
 (1, "Dünya nüfusu 2084'te 10,29 milyarla zirve", "Destek", "Lider", ok if ctrl.uyum.iloc[:2].all() else "Fark var"),
 (2, "Afrika'nın payı %9,1 → %18,6 → %37,5",     "Destek", "Lider", ok if ctrl.uyum.iloc[2:5].all() else "Fark var"),
 (3, "2100'de en yaşlı bölge Latin Amerika",      "Destek", "Lider", ok if (kaynak == "bm_bolge_satiri" and oldest_2100.startswith("Latin")) else "TASLAK: BM bölge satırıyla tekrar hesapla"),
], columns=["#", "sayı", "hesaplayan", "kontrol eden", "durum"])
display(kontrol_listesi)

2100'de en yaşlı bölge: Latin America and the Caribbean

BULGU (g01 A):  Afrika'nın dünya nüfusundaki payı %9.1 (1950) → %18.6 (2024) → %37.5 (2100).
BULGU (g01 B):  2100'de en yaşlı bölge Latin America and the Caribbean (kaynak: bm_bolge_satiri).


,#,sayı,hesaplayan,kontrol eden,durum
0,1,"Dünya nüfusu 2084'te 10,29 milyarla zirve",Destek,Lider,"Hesaplandı, kontrol bekliyor"
1,2,"Afrika'nın payı %9,1 → %18,6 → %37,5",Destek,Lider,"Hesaplandı, kontrol bekliyor"
2,3,2100'de en yaşlı bölge Latin Amerika,Destek,Lider,"Hesaplandı, kontrol bekliyor"
